In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, accuracy_score
from sklearn.ensemble import RandomForestClassifier


# 5. Implementation Tasks
## 5.1  Data Pre-processing
#### 1. Load the provided user and article datasets.

In [2]:
train_users = pd.read_csv(r'C:\Users\saksh\Desktop\RL\lab3\lab3-contextual-bandit\data\train_users.csv')
test_users = pd.read_csv(r'C:\Users\saksh\Desktop\RL\lab3\lab3-contextual-bandit\data\test_users.csv')
news_articles = pd.read_csv(r'C:\Users\saksh\Desktop\RL\lab3\lab3-contextual-bandit\data\news_articles.csv')

In [3]:
train_users.head()

,user_id,age,income,clicks,purchase_amount,session_duration,content_variety,engagement_score,num_transactions,avg_monthly_spend,...,screen_brightness,battery_percentage,cart_abandonment_count,browser_version,background_app_count,session_inactivity_duration,network_jitter,region_code,subscriber,label
0,U7392,NaN,23053,10,500.00,17.34,0.36661,37.29781,3,187.44,...,4.0,2.0,8,3.17.97,10,22.75,8.0,Z999,False,user_3
1,U2702,56.0,20239,11,913.33,22.22,0.61370,59.36342,5,145.15,...,4.5,63.0,5,1.57.10,8,1.75,4.0,U428,True,user_2
2,U2461,NaN,13907,9,1252.62,41.57,0.80368,76.78706,7,282.03,...,1.3,22.0,2,2.16.94,12,29.33,18.0,Z999,True,user_3
3,U7475,NaN,26615,12,500.00,30.17,0.26499,30.19441,10,195.35,...,4.2,77.0,9,9.90.20,4,21.61,22.0,X123,False,user_3
4,U6040,32.0,27958,13,500.00,65.27,0.36385,37.12153,5,439.68,...,4.6,30.0,9,1.99.38,7,7.58,52.0,S043,False,user_1


In [4]:
test_users.head()

,user_id,age,income,clicks,purchase_amount,session_duration,content_variety,engagement_score,num_transactions,avg_monthly_spend,...,loyalty_index,screen_brightness,battery_percentage,cart_abandonment_count,browser_version,background_app_count,session_inactivity_duration,network_jitter,region_code,subscriber
0,U4058,36.0,10000,4,500.00,2.54,0.47162,49.32668,8,229.80,...,13.0,3.0,39.0,3,5.99.65,7,7.86,26.0,X789,True
1,U1118,33.0,19607,8,1906.81,39.74,0.55219,56.42526,4,819.43,...,30.0,4.6,84.0,5,7.22.85,12,2.79,42.0,P752,True
2,U6555,51.0,21049,11,500.00,46.07,0.69961,58.88307,5,113.35,...,21.0,2.1,88.0,7,3.29.28,11,27.50,11.0,P878,False
3,U9170,58.0,25752,13,500.00,60.86,0.86843,86.15690,8,237.13,...,37.0,1.9,34.0,7,6.80.77,15,22.21,34.0,Z670,False
4,U3348,32.0,25302,12,2693.22,39.77,0.81805,84.82418,5,1491.56,...,45.0,3.0,6.0,6,9.29.14,8,11.44,0.0,M101,False


In [5]:
news_articles.head()

,link,headline,category,short_description,authors,date
0,https://www.huffpost.com/entry/covid-boosters-...,Over 4 Million Americans Roll Up Sleeves For O...,U.S. NEWS,Health experts said it is too early to predict...,"Carla K. Johnson, AP",2022-09-23
1,https://www.huffpost.com/entry/american-airlin...,"American Airlines Flyer Charged, Banned For Li...",U.S. NEWS,He was subdued by passengers and crew when he ...,Mary Papenfuss,2022-09-23
2,https://www.huffpost.com/entry/funniest-tweets...,23 Of The Funniest Tweets About Cats And Dogs ...,COMEDY,"""Until you have a dog you don't understand wha...",Elyse Wanshel,2022-09-23
3,https://www.huffpost.com/entry/funniest-parent...,The Funniest Tweets From Parents This Week (Se...,PARENTING,"""Accidentally put grown-up toothpaste on my to...",Caroline Bologna,2022-09-23
4,https://www.huffpost.com/entry/amy-cooper-lose...,Woman Who Called Cops On Black Bird-Watcher Lo...,U.S. NEWS,Amy Cooper accused investment firm Franklin Te...,Nina Golgowski,2022-09-22


In [6]:
print("Train Users Shape:", train_users.shape)
print("News Articles Shape:", news_articles.shape)
train_users['label'].unique()

Train Users Shape: (2000, 33)
News Articles Shape: (209527, 6)


array(['user_3', 'user_2', 'user_1'], dtype=object)

### 5.1
#### 2. Perform necessary data cleaning

In [7]:
train_users.isnull().sum()

user_id                          0
age                            698
income                           0
clicks                           0
purchase_amount                  0
session_duration                 0
content_variety                  0
engagement_score                 0
num_transactions                 0
avg_monthly_spend                0
avg_cart_value                   0
browsing_depth                   0
revisit_rate                     0
scroll_activity                  0
time_on_site                     0
interaction_count                0
preferred_price_range            0
discount_usage_rate              0
wishlist_size                    0
product_views                    0
repeat_purchase_gap (days)       0
churn_risk_score                 0
loyalty_index                    0
screen_brightness                0
battery_percentage               0
cart_abandonment_count           0
browser_version                  0
background_app_count             0
session_inactivity_d

In [8]:
train_users["age"] = train_users["age"].fillna(train_users["age"].median())
X = train_users.drop(columns=["label", "user_id"])
y = train_users["label"]


### 5.1
#### 3. Apply feature encoding where required to prepare the data for classification and bandit training.

In [9]:
X_encoded = pd.get_dummies(X, columns=["browser_version", "region_code"], drop_first=True)
X_encoded["subscriber"] = X_encoded["subscriber"].astype(int)

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
label_encoder.classes_

array(['user_1', 'user_2', 'user_3'], dtype=object)

### 5.2 User Classification
#### • Split the train_users.csv dataset into a training set (80%) and a validation set (20%) for model evaluation.


In [10]:
X_train, X_val, y_train, y_val = train_test_split(
    X_encoded,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

### 5.2
#### The model must be trained on the training set and evaluated on the validation set to ensure it can accurately classify users into their respective categories.


In [16]:
rf_clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

rf_clf.fit(X_train, y_train)

y_val_pred = rf_clf.predict(X_val)

print("Validation Accuracy:", accuracy_score(y_val, y_val_pred))
print("\nClassification Report:\n")
print(classification_report(
    y_val,
    y_val_pred,
    target_names=label_encoder.classes_
))

Validation Accuracy: 0.8775

Classification Report:

              precision    recall  f1-score   support

      user_1       0.87      0.85      0.86       142
      user_2       0.91      0.89      0.90       142
      user_3       0.84      0.89      0.87       116

    accuracy                           0.88       400
   macro avg       0.88      0.88      0.88       400
weighted avg       0.88      0.88      0.88       400

